In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
from scipy.stats import multivariate_normal
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# Dataset and Setup


## Dataset: Handwritten Digits

In [2]:
digits = load_digits()      
X = digits.data            
y = digits.target

## Data splitting

In [3]:

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=0)
X_val, X_test, y_val, y_test = train_test_split( X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=0)


#Standardize the features

scaler = StandardScaler()
scaler.fit(X_train)

X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
classes=10



## Gaussian Generative Model

In [4]:

#  Class priors 
def class_priors(y):

    pi_K = np.array([np.mean(y == k) for k in range(classes)])
    return pi_K


# Class means 
def class_means(X, y):
    M_K = np.array([X[y == k].mean(axis=0) for k in range(classes)])
    return M_K


# Shared covariance 
def shared_covariance(X, y, M_K):
    n_features = X.shape[1]
    Sigma = np.zeros((n_features, n_features))

    for k in range(classes):
        X_k = X[y == k]
        diffs = X_k - M_K[k]
        Sigma += diffs.T @ diffs

    Sigma /= X.shape[0]
    return Sigma


#  Regularization
def regularize_covariance(Sigma, h):
    Sigma_h = Sigma + h * np.eye(Sigma.shape[0])
    return Sigma_h


# Prediction rule
def predict(X, pi_K, M_K, Sigma_h):
    
    n_samples = X.shape[0]
    scores = np.zeros((n_samples,classes))

    mvn_list = [
        multivariate_normal(mean=M_K[k], cov=Sigma_h)
        for k in range(classes)
    ]

    for k in range(classes):
        scores[:, k] = np.log(pi_K[k]) + mvn_list[k].logpdf(X)

    y_pred = np.argmax(scores, axis=1)
    return y_pred


##  Hyperparameter Tuning and Evaluation

In [5]:
lambdas = [1e-4, 1e-3, 1e-2, 1e-1]

pi_K = class_priors(y_train)
M_K = class_means(X_train_s, y_train)
Sigma = shared_covariance(X_train_s, y_train, M_K)

best_lambda = None
best_acc = 0

for h in lambdas:
    Sigma_h = regularize_covariance(Sigma,h)
    
    y_pred_val = predict(X_val_s, pi_K, M_K, Sigma_h)
    acc = np.mean(y_pred_val == y_val)
    
    print(f"λ={h}, Validation Accuracy={acc:.4f}")
    
    # if acc > best_acc:
    #     best_acc = acc
    #     best_lambda = h


    

λ=0.0001, Validation Accuracy=0.9444
λ=0.001, Validation Accuracy=0.9444
λ=0.01, Validation Accuracy=0.9407
λ=0.1, Validation Accuracy=0.9370


### Combine the training and validation sets into a single training set

In [6]:
best_lambda = .001
X_final_train = np.vstack([X_train_s, X_val_s])
y_final_train = np.hstack([y_train, y_val])

pi_K_final =class_priors(y_final_train)
M_K_final = class_means(X_final_train, y_final_train)
Sigma_final =shared_covariance(X_final_train, y_final_train, M_K_final)
Sigma_h_final = regularize_covariance(Sigma_final, best_lambda)

y_pred_test = predict(X_test_s, pi_K_final, M_K_final, Sigma_h_final)
accuracy = np.mean(y_pred_test == y_test)
print("Test Accuracy:", accuracy)


Test Accuracy: 0.9777777777777777


In [7]:

precision = precision_score(y_test, y_pred_test, average='macro')
recall = recall_score(y_test, y_pred_test, average='macro')
f1 = f1_score(y_test, y_pred_test, average='macro')
cm = confusion_matrix(y_test, y_pred_test)

print("Macro-averaged Precision:", precision)
print("Macro-averaged Recall:", recall)
print("Macro-averaged F1-score:", f1)
print("Confusion Matrix:\n", cm)


Macro-averaged Precision: 0.9793437446663253
Macro-averaged Recall: 0.9776353276353277
Macro-averaged F1-score: 0.9779301638559714
Confusion Matrix:
 [[27  0  0  0  0  0  0  0  0  0]
 [ 0 28  0  0  0  0  0  0  0  0]
 [ 0  0 27  0  0  0  0  0  0  0]
 [ 0  0  0 27  0  0  0  0  0  0]
 [ 0  0  0  0 26  0  0  0  1  0]
 [ 0  0  0  0  0 26  0  0  1  0]
 [ 0  2  0  0  0  0 25  0  0  0]
 [ 0  0  0  0  0  0  0 27  0  0]
 [ 0  1  0  0  0  0  0  0 25  0]
 [ 0  0  0  1  0  0  0  0  0 26]]


### Explanation of the Generative Model

 In this model, we assume a probabilistic process that generates the data.
   
- First, the class label y is drawn from a categorical distribution with class priors:
𝑝(𝑦=𝑘)=pi_𝑘,     𝑘=0,1,…,9.

Given the class label, the input vector 𝑥∈𝑅^64 is assumed to be generated from a multivariate Gaussian distribution that shares the same covariance matrix across all classes:

𝑝(𝑥∣𝑦=𝑘)=𝑁(𝑥;M_K,Σ).


-To train the model, we estimate the parameters from the training data:

* Class priors pi_K :are computed as the proportion of training samples belonging to each class.

* Class means M_K: are computed by averaging all training examples from class k.

* The shared covariance matrix :is computed by accumulating

(𝑥𝑖−M𝑦𝑖)(𝑥𝑖−M𝑦𝑖)^𝑇  over all training points and dividing by the number of samples.

- Since the empirical covariance matrix may be poorly conditioned or nearly singular, we apply regularization:

sigma𝜆=sigma+𝜆𝐼.
The parameter λ controls how much smoothing is applied.
Small λ may lead to overfitting and numerical instability, while large λ oversmooths the covariance and can cause underfitting.


* The main confusions appear between 4–8, 5–8, 6–1, and 9–3, mainly due to similarity in some handwritten samples.

* Effect of λ caused numerical instability, while large 
λ reduced accuracy. Medium values provided the most stable and accurate performance.

* The model is simple, fast, and works well when class covariances are similar. However, its Gaussian and linear assumptions limit its ability to fully capture the complexity of handwritten digits


